In [3]:
# ============================================================
# AGRIVISION AI
# FULL PLANTVILLAGE + PRETRAINED MOBILENETV2 + TFLITE
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL KAGGLEHUB
# ------------------------------------------------------------

!pip install -q kagglehub


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import tensorflow as tf
import kagglehub

from google.colab import files


print("TensorFlow Version:", tf.__version__)

gpu = tf.config.list_physical_devices("GPU")

if gpu:
    print("✅ GPU detected:", gpu)
else:
    print("⚠️ GPU NOT detected. Training will be slower.")


# ------------------------------------------------------------
# 3. DOWNLOAD FULL PLANTVILLAGE DATASET
# ------------------------------------------------------------

print("\nDownloading full PlantVillage dataset...")

path = kagglehub.dataset_download(
    "abdallahalidev/plantvillage-dataset"
)

print("Downloaded at:")
print(path)


# ------------------------------------------------------------
# 4. AUTOMATICALLY FIND 38-CLASS DATASET FOLDER
# ------------------------------------------------------------

DATASET_PATH = None

for root, dirs, file_names in os.walk(path):

    # We want the directory containing the 38 disease folders
    if len(dirs) == 38:

        DATASET_PATH = root
        break


if DATASET_PATH is None:

    # Backup search
    for root, dirs, file_names in os.walk(path):

        if len(dirs) >= 38:
            DATASET_PATH = root
            break


if DATASET_PATH is None:
    raise Exception(
        "❌ Could not automatically find the 38-class PlantVillage folder."
    )


print("\n✅ Dataset folder found:")
print(DATASET_PATH)


# ------------------------------------------------------------
# 5. CHECK CLASSES
# ------------------------------------------------------------

folders = sorted([
    folder
    for folder in os.listdir(DATASET_PATH)
    if os.path.isdir(
        os.path.join(DATASET_PATH, folder)
    )
])


print("\nTotal folders found:", len(folders))

print("\nClasses:")

for i, name in enumerate(folders):
    print(i, ":", name)


# ------------------------------------------------------------
# 6. DATASET SETTINGS
# ------------------------------------------------------------

IMG_SIZE = (224, 224)

# 64 is faster on GPU.
# If memory error occurs, change to 32.
BATCH_SIZE = 64


# ------------------------------------------------------------
# 7. CREATE TRAINING DATASET
# ------------------------------------------------------------

print("\nCreating training dataset...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,

    validation_split=0.20,

    subset="training",

    seed=123,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE
)


# ------------------------------------------------------------
# 8. CREATE VALIDATION DATASET
# ------------------------------------------------------------

print("\nCreating validation dataset...")

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,

    validation_split=0.20,

    subset="validation",

    seed=123,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE
)


# ------------------------------------------------------------
# 9. GET CLASS NAMES
# ------------------------------------------------------------

class_names = train_ds.class_names

NUM_CLASSES = len(class_names)


print("\n================================")
print("TOTAL CLASSES:", NUM_CLASSES)
print("================================")


for i, name in enumerate(class_names):
    print(i, "->", name)


if NUM_CLASSES != 38:

    print(
        "\n⚠️ WARNING: Expected 38 classes but found",
        NUM_CLASSES
    )


# ------------------------------------------------------------
# 10. SPEED UP DATA LOADING
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(
    buffer_size=AUTOTUNE
)

val_ds = val_ds.prefetch(
    buffer_size=AUTOTUNE
)


# ------------------------------------------------------------
# 11. DATA AUGMENTATION
# ------------------------------------------------------------

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip(
        "horizontal"
    ),

    tf.keras.layers.RandomRotation(
        0.10
    ),

    tf.keras.layers.RandomZoom(
        0.10
    )

])


# ------------------------------------------------------------
# 12. LOAD PRETRAINED MOBILENETV2
# ------------------------------------------------------------

print("\nLoading pretrained MobileNetV2...")

base_model = tf.keras.applications.MobileNetV2(

    input_shape=(
        224,
        224,
        3
    ),

    include_top=False,

    # IMPORTANT:
    # Use pretrained ImageNet weights
    weights="imagenet"
)


# ------------------------------------------------------------
# 13. FREEZE PRETRAINED MODEL
# ------------------------------------------------------------

# This makes training MUCH faster.

base_model.trainable = False


print("✅ MobileNetV2 loaded.")
print("✅ Pretrained layers frozen.")


# ------------------------------------------------------------
# 14. CREATE AGRIVISION MODEL
# ------------------------------------------------------------

model = tf.keras.Sequential([

    # Input
    tf.keras.layers.Input(
        shape=(224, 224, 3)
    ),


    # Data augmentation
    data_augmentation,


    # MobileNetV2 expects values from -1 to +1
    tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1
    ),


    # Pretrained CNN
    base_model,


    # Convert feature maps into vector
    tf.keras.layers.GlobalAveragePooling2D(),


    # Reduce overfitting
    tf.keras.layers.Dropout(
        0.20
    ),


    # Final disease classification
    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )

])


# ------------------------------------------------------------
# 15. COMPILE MODEL
# ------------------------------------------------------------

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)


# ------------------------------------------------------------
# 16. DISPLAY MODEL
# ------------------------------------------------------------

print("\nAGRIVISION MODEL:")

model.summary()


# ------------------------------------------------------------
# 17. CALLBACKS
# ------------------------------------------------------------

callbacks = [

    # Stop automatically if validation stops improving
    tf.keras.callbacks.EarlyStopping(

        monitor="val_accuracy",

        patience=2,

        restore_best_weights=True
    ),


    # Save best model
    tf.keras.callbacks.ModelCheckpoint(

        "best_agrivision.keras",

        monitor="val_accuracy",

        save_best_only=True
    )
]


# ------------------------------------------------------------
# 18. TRAIN MODEL
# ------------------------------------------------------------

print("\n================================")
print("🚀 STARTING TRAINING")
print("================================")


history = model.fit(

    train_ds,

    validation_data=val_ds,

    # 5 epochs = fast training
    epochs=5,

    callbacks=callbacks
)


# ------------------------------------------------------------
# 19. TEST FINAL ACCURACY
# ------------------------------------------------------------

print("\nEvaluating model...")

loss, accuracy = model.evaluate(
    val_ds
)


print("\n================================")
print("🎯 FINAL RESULTS")
print("================================")

print(
    "Validation Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print(
    "Validation Loss:",
    round(loss, 4)
)


# ------------------------------------------------------------
# 20. SAVE CLASS LABELS
# ------------------------------------------------------------

with open(
    "labels.txt",
    "w"
) as f:

    for name in class_names:

        f.write(
            name + "\n"
        )


print("\n✅ labels.txt created.")


# ------------------------------------------------------------
# 21. SAVE KERAS MODEL
# ------------------------------------------------------------

model.save(
    "agrivision_model.keras"
)

print(
    "✅ Keras model saved."
)


# ------------------------------------------------------------
# 22. CONVERT MODEL TO TFLITE
# ------------------------------------------------------------

print(
    "\nConverting to TensorFlow Lite..."
)


converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)


# Optimize model size/performance
converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]


tflite_model = converter.convert()


# ------------------------------------------------------------
# 23. SAVE TFLITE MODEL
# ------------------------------------------------------------

TFLITE_FILE = "agrivision_model.tflite"


with open(
    TFLITE_FILE,
    "wb"
) as f:

    f.write(
        tflite_model
    )


model_size = (
    os.path.getsize(TFLITE_FILE)
    / 1024
    / 1024
)


print("\n================================")
print("✅ TFLITE MODEL CREATED")
print("================================")

print(
    "File:",
    TFLITE_FILE
)

print(
    "Size:",
    round(model_size, 2),
    "MB"
)


# ------------------------------------------------------------
# 24. TEST TFLITE MODEL
# ------------------------------------------------------------

print(
    "\nTesting TFLite model..."
)


interpreter = tf.lite.Interpreter(
    model_path=TFLITE_FILE
)


interpreter.allocate_tensors()


input_details = (
    interpreter.get_input_details()
)

output_details = (
    interpreter.get_output_details()
)


print(
    "\nInput shape:",
    input_details[0]["shape"]
)

print(
    "Output shape:",
    output_details[0]["shape"]
)


print(
    "\n✅ TFLite model loads successfully!"
)


# ------------------------------------------------------------
# 25. SHOW FINAL INFORMATION
# ------------------------------------------------------------

print("\n================================")
print("🌱 AGRIVISION AI COMPLETE")
print("================================")

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

print(
    "Model:",
    "agrivision_model.tflite"
)

print(
    "Labels:",
    "labels.txt"
)


# ------------------------------------------------------------
# 26. DOWNLOAD FILES
# ------------------------------------------------------------

print(
    "\nDownloading files..."
)


files.download(
    "agrivision_model.tflite"
)


files.download(
    "labels.txt"
)

TensorFlow Version: 2.20.0
✅ GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Downloaded at:
/kaggle/input/plantvillage-dataset

✅ Dataset folder found:
/kaggle/input/plantvillage-dataset/plantvillage dataset/segmented

Total folders found: 38

Classes:
0 : Apple___Apple_scab
1 : Apple___Black_rot
2 : Apple___Cedar_apple_rust
3 : Apple___healthy
4 : Blueberry___healthy
5 : Cherry_(including_sour)___Powdery_mildew
6 : Cherry_(including_sour)___healthy
7 : Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 : Corn_(maize)___Common_rust_
9 : Corn_(maize)___Northern_Leaf_Blight
10 : Corn_(maize)___healthy
11 : Grape___Black_rot
12 : Grape___Esca_(Black_Measles)
13 : Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 : Grape___healthy
15 : Orange___Haunglongbing_(Citrus_greening)
16 : Peach___Bacterial_spot
17 : Peach___healthy
18 : Pepper,_bell___Bacterial_spot
19 : Pepper,_bell___health

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        48,678 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,306,662 (8.80 MB)

 Trainable params: 48,678 (190.15 KB)

...

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>